# Projet 1 : Mastermind — Explications

Ce notebook explique **pourquoi** la solution de `TP_resolution.ipynb` fonctionne, étape par étape : les concepts utilisés, les liens avec les leçons, des variantes et les erreurs à éviter.

Les petites cellules de code ici **n'ouvrent pas de fenêtre** : elles testent la logique du jeu seule, et on peut les exécuter partout (y compris dans Colab).

### Plan
0. Organisation générale du programme
1. Étape 1 — L'interface et le tableau à deux dimensions
2. Étape 2 — Pseudo et combinaison secrète
3. Étape 3 — Changer la couleur d'un bouton
4. Étapes 4 et 5 — Compter les pions bien / mal placés
5. Étape 6 — Un tour de jeu et les variables globales
6. Étape 7 — Rejouer
7. Étape 8 — Fichier CSV et `try / except`
8. Étape 9 — Tri à bulle et classement
9. Étape 10 — Affichage du classement

---
## 0. Organisation générale du programme

Un jeu Tkinter ne s'écrit pas comme un programme « qui se déroule de haut en bas ». On distingue :

1. **La mise en place** : on crée la fenêtre, on dessine le plateau et on crée les boutons. Ce code est exécuté une seule fois.
2. **Les fonctions « réactions »** (`action1`, `verification`, `rejouer`…) : elles sont reliées aux boutons par `command=` et Tkinter les appelle **à chaque clic**.
3. **La boucle d'événements** `fenetre.mainloop()` : elle attend les clics. Elle doit être **la dernière instruction** du programme.

**L'état du jeu** est conservé dans quelques variables :

| Variable | Contenu |
|:---|:---|
| `combinaison` | la liste des 4 couleurs secrètes |
| `Choix` | les 4 couleurs actuellement affichées sur les boutons |
| `numero_essai` | la ligne en cours (0 à 11) |
| `partie_terminee` | `True` / `False` |
| `Grille`, `Secret`, `Textes_bien`, `Textes_mal` | les **identifiants** des dessins du canevas |

⚠️ **Erreur courante** : mettre `fenetre.mainloop()` au milieu du code. Tout ce qui est écrit après ne s'exécute qu'à la fermeture de la fenêtre.

---
## 1. Étape 1 — L'interface et le tableau à deux dimensions

### L'idée clé : conserver les identifiants
`canevas.create_oval(...)` dessine un rond **et renvoie un nombre**, son identifiant. Il faut ranger ce nombre pour pouvoir plus tard écrire `canevas.itemconfig(identifiant, fill='red')`.

La grille a 12 lignes et 4 colonnes, donc on range les identifiants dans une **liste de listes**. On construit ce tableau avec `append` (leçon 02B) :
- `Grille.append([])` ajoute une nouvelle ligne vide ;
- `Grille[ligne].append(objet)` ajoute un identifiant au bout de cette ligne.

On accède ensuite au rond de la ligne `l` et de la colonne `c` avec `Grille[l][c]`.

Voici la même construction, avec de faux identifiants (un compteur) à la place du canevas :

In [1]:
# Simulation : un compteur remplace les identifiants renvoyés par create_oval
compteur = 0
Grille = []
for ligne in range(3):
    Grille.append([])                   # nouvelle ligne
    for colonne in range(4):
        compteur = compteur + 1         # "identifiant" du rond
        Grille[ligne].append(compteur)
print(Grille)
print("Rond ligne 1, colonne 2 :", Grille[1][2])

[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
Rond ligne 1, colonne 2 : 7


### Les positions : de (ligne, colonne) à (x, y)
Le centre d'un rond est calculé à partir de sa ligne et de sa colonne :

```python
x = X_GRILLE + colonne * ECART     # on se décale vers la droite à chaque colonne
y = Y_GRILLE + ligne * HAUTEUR     # on descend à chaque ligne
create_oval(x - RAYON, y - RAYON, x + RAYON, y + RAYON)   # carré englobant le rond
```

On utilise des **constantes** en majuscules (`ECART`, `RAYON`…) : changer une seule valeur redimensionne toute la grille.

### Variantes
- On peut dessiner la grille **de bas en haut** (le 1er essai en bas, comme sur un vrai plateau) avec `y = 470 - ligne * HAUTEUR`.
- On aurait aussi pu faire les 4 « boutons de couleur » avec des ronds du canevas, mais le sujet demande des **boutons**.

### Erreurs courantes
- ❌ `canevas.create_oval(...)` sans stocker le résultat : le rond est dessiné mais on ne pourra plus jamais le modifier.
- ❌ `Grille = [[]] * 12` : les 12 lignes seraient **la même liste** ! Il faut faire `append([])` dans la boucle.
- ❌ Oublier `canevas.grid(...)` (ou `pack()`) : le widget existe mais ne s'affiche pas.

---
## 2. Étape 2 — Pseudo et combinaison secrète

### Le pseudo
Une `Entry` est une zone de saisie. On lit son contenu avec `entree_pseudo.get()`, qui renvoie une chaîne, éventuellement vide `""`.

### La combinaison
On tire 4 fois une couleur au hasard avec `choice(COULEURS)`. C'est la fonction du tableau « Fonctions particulières » de la **leçon 02B** (module random).
Comme chaque tirage est indépendant, la même couleur peut sortir plusieurs fois : c'est bien « avec répétitions possibles ».

In [2]:
from random import choice, shuffle
COULEURS = ['white', 'black', 'yellow', 'blue', 'red', 'green']

def generer_combinaison():
    combinaison = []
    for i in range(4):
        combinaison.append(choice(COULEURS))
    return combinaison

for k in range(3):
    print(generer_combinaison())

['white', 'red', 'yellow', 'yellow']
['yellow', 'yellow', 'blue', 'blue']
['white', 'blue', 'yellow', 'blue']


### Erreur classique : `shuffle` ou `COULEURS[:4]`
Mélanger puis prendre les 4 premières (`shuffle(COULEURS)` puis `COULEURS[:4]`) donne 4 couleurs **toutes différentes**. Or le sujet autorise les répétitions : ce serait une combinaison plus facile à deviner. (C'était la bonne méthode pour le **tiercé** de la leçon 02B, où les numéros devaient être distincts.)

`print(combinaison)` au lancement permet de **tricher pour tester** les étapes suivantes. On pourra l'enlever à la fin.

---
## 3. Étape 3 — Changer la couleur d'un bouton

### Passer à la couleur suivante
1. On cherche la position de la couleur actuelle avec `COULEURS.index(...)` (**leçon 02B** : « renvoie l'indice du premier élément ayant la valeur e »).
2. On ajoute 1, sauf si c'était la dernière couleur : on revient alors à l'indice 0 (« on boucle »).

In [3]:
def couleur_suivante(couleur):
    indice = COULEURS.index(couleur)
    if indice == len(COULEURS) - 1:
        indice = 0
    else:
        indice = indice + 1
    return COULEURS[indice]

c = 'white'
for k in range(7):
    print(c, end=" → ")
    c = couleur_suivante(c)
print(c)

white → black → yellow → blue → red → green → white → black


### Pourquoi `action1`, `action2`… ?
`command=` attend **le nom d'une fonction sans parenthèses** :
- ✅ `Boutons[0].config(command=action1)` : Tkinter appellera `action1()` au clic ;
- ❌ `command=changer_couleur(0)` : la fonction est **exécutée tout de suite**, une seule fois, et c'est son résultat (`None`) qui est donné à `command`.

Sans `lambda` (non vue en cours), il faut donc une petite fonction sans paramètre par bouton, qui appelle `changer_couleur` avec le bon numéro.

### Pourquoi pas de `global` pour `Choix` ?
`Choix[numero_bouton] = ...` **modifie un élément** de la liste : la variable `Choix` désigne toujours la même liste. Python n'a besoin de `global` que si on **réaffecte** la variable elle-même (`Choix = ...`).

### `recuperer_proposition()` renvoie une copie
On construit une nouvelle liste. Si on renvoyait `Choix` directement, la « proposition » changerait dès que le joueur clique à nouveau sur un bouton.

---
## 4. Étapes 4 et 5 — Compter les pions bien / mal placés

C'est le **cœur logique** du jeu. On le teste sans interface.

### Bien placés
On compare les deux listes **indice par indice** (`Liste[i]`, leçon 02B).

In [4]:
def compter_bien_places(proposition, combinaison):
    nb = 0
    for i in range(4):
        if proposition[i] == combinaison[i]:
            nb = nb + 1
    return nb

print(compter_bien_places(['red','red','blue','green'], ['red','red','blue','green']))   # 4
print(compter_bien_places(['red','blue','red','blue'], ['red','red','blue','green']))    # 1

4
1


### Mal placés : le piège du double comptage
Idée naïve : compter les couleurs de la proposition présentes dans la combinaison. **C'est faux** :
- proposition `['red','red','red','red']` / combinaison `['red','blue','blue','blue']`
- la méthode naïve trouve 4 pions « présents », alors qu'il n'y a **qu'un** rouge dans la combinaison, et qu'il est déjà **bien placé** !

La méthode du sujet règle les deux problèmes :
1. **On retire les positions bien placées** des deux listes, ce qui donne `proposition_restante` et `combinaison_restante`.
2. **Chaque couleur trouvée est retirée** de `combinaison_restante` avec `remove` (leçon 02B : supprime **la première occurrence**). Elle ne peut donc servir qu'une fois.

In [5]:
def compter_mal_places(proposition, combinaison, detail=False):
    combinaison_restante = []
    proposition_restante = []
    for i in range(4):
        if proposition[i] != combinaison[i]:
            combinaison_restante.append(combinaison[i])
            proposition_restante.append(proposition[i])
    if detail:
        print("  proposition_restante :", proposition_restante)
        print("  combinaison_restante :", combinaison_restante)
    nb = 0
    for couleur in proposition_restante:
        if couleur in combinaison_restante:
            nb = nb + 1
            combinaison_restante.remove(couleur)
            if detail:
                print("  ", couleur, "trouvé → nb =", nb, "| il reste", combinaison_restante)
    return nb

print("Exemple du sujet :")
print(compter_mal_places(['blue','blue','green','red'], ['red','red','blue','green'], detail=True))   # 3
print("Cas piège :")
print(compter_mal_places(['red','red','red','red'], ['red','blue','blue','blue'], detail=True))      # 0

Exemple du sujet :
  proposition_restante : ['blue', 'blue', 'green', 'red']
  combinaison_restante : ['red', 'red', 'blue', 'green']
   blue trouvé → nb = 1 | il reste ['red', 'red', 'green']
   green trouvé → nb = 2 | il reste ['red', 'red']
   red trouvé → nb = 3 | il reste ['red']
3
Cas piège :
  proposition_restante : ['red', 'red', 'red']
  combinaison_restante : ['blue', 'blue', 'blue']
0


Dans l'exemple du sujet, le 2ᵉ `'blue'` n'est **pas** compté : le seul bleu de la combinaison a déjà été « consommé » par le premier.

### Vérification rapide avec `assert`
`assert condition` ne fait rien si la condition est vraie et provoque une erreur sinon. C'est pratique pour tester.

In [6]:
assert compter_bien_places(['red','red','blue','green'], ['red','red','blue','green']) == 4
assert compter_mal_places(['blue','blue','green','red'], ['red','red','blue','green']) == 3
assert compter_mal_places(['red','red','red','red'], ['red','blue','blue','blue']) == 0
assert compter_mal_places(['blue','red','white','white'], ['red','blue','black','black']) == 2
print("Tous les tests passent ✔")

Tous les tests passent ✔


### Erreurs courantes
- ❌ Faire `remove` sur la **vraie** liste `combinaison` : la combinaison secrète serait détruite pour les tours suivants. On travaille sur des **copies** (les listes « restantes »).
- ❌ Oublier l'étape 1 : les pions bien placés sont alors aussi comptés comme mal placés.

---
## 5. Étape 6 — Un tour de jeu et les variables globales

### Le déroulement de `verification()`
```
partie terminée ?  ── oui ──► return (on ne fait rien)
      │ non
pseudo vide ?      ── oui ──► message d'erreur, return (essai non compté)
      │ non
afficher pseudo, récupérer la proposition, colorier la ligne numero_essai,
calculer bien / mal, afficher les 2 nombres
      │
bien == 4 ?        ── oui ──► révéler, message victoire, fin, enregistrer le score
      │ non
dernier essai ?    ── oui ──► révéler, message défaite, fin
      │
numero_essai + 1
```

Le `return` placé tôt arrête la fonction immédiatement : c'est ce qui « ne compte pas » la tentative, puisqu'on n'arrive jamais à `numero_essai = numero_essai + 1`.

### Pourquoi `global` ?
`numero_essai` et `partie_terminee` sont **réaffectées** dans la fonction (`numero_essai = numero_essai + 1`). Sans `global`, Python les considère comme des variables **locales** et on obtient :
`UnboundLocalError: local variable 'numero_essai' referenced before assignment`.

### Le score
Les essais sont numérotés à partir de 0. Au 1er essai, `numero_essai` vaut 0, donc on a utilisé `numero_essai + 1` essais. C'est pour cela que le sujet demande `score = numero_essai + 1`, **avant** d'incrémenter.

### Condition de défaite
Le dernier essai est la ligne d'indice `NB_ESSAIS - 1 = 11`. Si on testait `numero_essai == 12`, on sortirait du tableau `Grille` (`IndexError`) avant de perdre.

### Erreur courante
- ❌ Tester la victoire **avant** de colorier la ligne : le joueur ne verrait pas sa dernière proposition.

---
## 6. Étape 7 — Rejouer

`rejouer()` doit remettre **tout** l'état du jeu dans sa configuration de départ. Une bonne méthode : reprendre la liste des variables de la partie 0 et remettre chacune à zéro :

| À réinitialiser | Comment |
|:---|:---|
| combinaison | `generer_combinaison()` (+ `print` pour tester) |
| grille | double boucle sur `Grille[ligne][colonne]` → `fill='lightgrey'` |
| historique bien / mal | `itemconfig(..., text="")` |
| combinaison cachée | ronds noirs + `"?"` réaffichés |
| compteur, fin de partie | `numero_essai = 0`, `partie_terminee = False` |
| messages | `message.config(text="")` |

On **ne recrée pas** les ronds : on modifie ceux qui existent grâce aux identifiants de l'étape 1. C'est tout l'intérêt de les avoir conservés !

- ❌ Oublier `combinaison` dans la ligne `global` : la nouvelle combinaison serait locale à `rejouer()` et le jeu continuerait avec l'ancienne.

---
## 7. Étape 8 — Fichier CSV et `try / except`

### Le format
```
Pseudo;Score        ← ligne d'en-tête (ignorée à la lecture)
Zelda;4
Link;6
```

### Lecture
- `readlines()` renvoie la liste des lignes, **avec** le `\n` final (sauf souvent la dernière ligne).
- `lignes[1:]` saute l'en-tête (tranche `Liste[p:]`, leçon 02B).
- On enlève le `\n` avec `ligne[:-1]` **seulement s'il est présent** (`ligne[-1] == "\n"`).
- `split(";")` découpe en `["Zelda", "4"]`. Le score est converti avec `int(...)` pour pouvoir **comparer des nombres** (sinon `"10" < "9"` serait vrai, car on comparerait des textes !).

### Écriture
On reconstruit chaque ligne avec `";".join(...)`, puis tout le texte avec `"\n".join(...)`. `join` n'accepte que des chaînes, d'où `str(score)`.

Test sur un fichier temporaire, grâce au paramètre par défaut `nom_fichier` :

In [7]:
FICHIER_SCORES = "scores_mastermind.csv"

def lire_classement(nom_fichier=FICHIER_SCORES):
    try:
        fichier = open(nom_fichier, "r", encoding="utf-8")
        lignes = fichier.readlines()
        fichier.close()
    except FileNotFoundError:
        return []
    classement = []
    for ligne in lignes[1:]:
        if ligne[-1] == "\n":
            ligne = ligne[:-1]
        if ligne != "":
            morceaux = ligne.split(";")
            classement.append([morceaux[0], int(morceaux[1])])
    return classement

def ecrire_classement(classement, nom_fichier=FICHIER_SCORES):
    lignes = ["Pseudo;Score"]
    for joueur in classement:
        lignes.append(";".join([joueur[0], str(joueur[1])]))
    texte = "\n".join(lignes)
    fichier = open(nom_fichier, "w", encoding="utf-8")
    fichier.write(texte)
    fichier.close()

# 1) Le fichier n'existe pas : try/except renvoie [] au lieu de planter
print(lire_classement("fichier_inexistant.csv"))

# 2) Écriture puis relecture
ecrire_classement([["Zelda", 4], ["Link", 6]], "test_scores.csv")
print(open("test_scores.csv", encoding="utf-8").read())
print(lire_classement("test_scores.csv"))

[]
Pseudo;Score
Zelda;4
Link;6
[['Zelda', 4], ['Link', 6]]


### Erreurs courantes
- ❌ Oublier `fichier.close()` : les données peuvent ne pas être écrites sur le disque.
- ❌ Oublier `encoding="utf-8"` : un pseudo avec accent (« Léa ») peut provoquer une erreur.
- ❌ `except:` sans préciser l'erreur : cela cacherait **toutes** les erreurs, même les bugs du programme. On attrape seulement `FileNotFoundError`, comme dans le cours.
- ❌ Ouvrir en `"a"` (ajout) au lieu de `"w"` : le fichier grossirait à chaque partie au lieu de ne garder que le top 3.

---
## 8. Étape 9 — Tri à bulle et classement

### Adapter le tri à bulle à une liste de listes
Dans le cours, on compare `liste[j] > liste[j + 1]`. Ici, chaque élément est un couple `[pseudo, score]`, donc :
- on **compare** les scores : `classement[j][1] > classement[j + 1][1]` ;
- on **échange** les couples entiers : `classement[j]` et `classement[j + 1]`. Il ne faut pas échanger seulement les scores, sinon les pseudos ne suivraient pas !

Le tri est « **sur place** » : la fonction modifie la liste reçue et ne renvoie rien.

In [8]:
def trier_classement(classement):
    n = len(classement)
    for i in range(n):
        for j in range(n - 1 - i):
            if classement[j][1] > classement[j + 1][1]:
                temporaire = classement[j]
                classement[j] = classement[j + 1]
                classement[j + 1] = temporaire

c = [["Mario", 9], ["Zelda", 4], ["Link", 6]]
trier_classement(c)
print(c)

[['Zelda', 4], ['Link', 6], ['Mario', 9]]


### Les règles du nouveau classement
On traduit l'énoncé **dans l'ordre** : une suite de « si… sinon si… sinon ».
1. un joueur a **exactement** ce score → on prend sa place ;
2. sinon, moins de 3 joueurs → on ajoute ;
3. sinon, meilleur (plus petit) que le **moins bon** → on le remplace. Après le tri, le moins bon est le **dernier**, `nouveau[-1]` (leçon 02B) ;
4. sinon → rien ne change.

On travaille sur une **copie** du classement, pour ne pas modifier la liste reçue en paramètre. Et on trie avant de renvoyer.

In [9]:
def calculer_nouveau_classement(classement, pseudo, score):
    nouveau = []
    for joueur in classement:
        nouveau.append([joueur[0], joueur[1]])
    trier_classement(nouveau)
    place_prise = False
    for i in range(len(nouveau)):
        if not place_prise and nouveau[i][1] == score:
            nouveau[i] = [pseudo, score]
            place_prise = True
    if not place_prise:
        if len(nouveau) < 3:
            nouveau.append([pseudo, score])
        elif score < nouveau[-1][1]:
            nouveau[-1] = [pseudo, score]
    trier_classement(nouveau)
    return nouveau

top = [["A", 5], ["B", 3], ["C", 7]]
print("Règle 1 (même score 5)  :", calculer_nouveau_classement(top, "D", 5))
print("Règle 2 (< 3 joueurs)   :", calculer_nouveau_classement([["A", 5]], "D", 8))
print("Règle 3 (4 bat 7)       :", calculer_nouveau_classement(top, "D", 4))
print("Règle 4 (9 trop mauvais):", calculer_nouveau_classement(top, "D", 9))
print("Liste d'origine intacte :", top)

Règle 1 (même score 5)  : [['B', 3], ['D', 5], ['C', 7]]
Règle 2 (< 3 joueurs)   : [['A', 5], ['D', 8]]
Règle 3 (4 bat 7)       : [['B', 3], ['D', 4], ['A', 5]]
Règle 4 (9 trop mauvais): [['B', 3], ['A', 5], ['C', 7]]
Liste d'origine intacte : [['A', 5], ['B', 3], ['C', 7]]


### Erreurs courantes
- ❌ Oublier que le **meilleur** score est le **plus petit** (moins d'essais) : il faut trier dans l'ordre croissant et comparer avec `<`.
- ❌ Faire `nouveau = classement` : ce n'est **pas** une copie, les deux noms désignent la même liste.
- ❌ Oublier le tri final : le nouveau joueur resterait à la fin même avec le meilleur score.

### Variante
`classement.sort()` (leçon 02B) sait trier des nombres, mais pas des listes de listes selon le 2ᵉ élément sans notions non vues. D'où le tri à bulle adapté, demandé par le sujet.

---
## 9. Étape 10 — Affichage du classement

On ne crée qu'**un seul** texte sur le canevas (`texte_classement`), dont on change le contenu avec `itemconfig(..., text=...)`. C'est le même principe que pour les ronds de l'étape 1.

On sépare le travail en petites fonctions qui ont chacune **un seul rôle** :

| Fonction | Rôle |
|:---|:---|
| `construire_texte_classement` | liste → texte (logique pure, testable) |
| `afficher_classement` | met le texte sur le canevas |
| `enregistrer_score_si_top3` | lire → calculer → écrire → afficher |

In [10]:
def construire_texte_classement(classement):
    texte = "Classement :"
    for i in range(len(classement)):
        texte = texte + "\n" + str(i + 1) + ". " + classement[i][0] + " : " + str(classement[i][1]) + " essai(s)"
    return texte

print(construire_texte_classement([["Zelda", 4], ["Link", 6], ["Mario", 9]]))

Classement :
1. Zelda : 4 essai(s)
2. Link : 6 essai(s)
3. Mario : 9 essai(s)


- La numérotation commence à `i + 1`, car les indices commencent à 0.
- `"\n"` passe à la ligne dans le texte affiché.
- ⚠️ On appelle `afficher_classement(lire_classement())` **au lancement**, sinon la zone reste vide jusqu'à la première victoire.

### Simulation d'une série de victoires (sans interface)
On enchaîne lire → calculer → écrire, comme le fait `enregistrer_score_si_top3` :

In [11]:
import os
fichier_test = "test_top3.csv"
if os.path.exists(fichier_test):
    os.remove(fichier_test)

for pseudo, score in [("Zelda", 6), ("Link", 4), ("Mario", 9), ("Peach", 5), ("Toad", 4)]:
    classement = lire_classement(fichier_test)
    nouveau = calculer_nouveau_classement(classement, pseudo, score)
    ecrire_classement(nouveau, fichier_test)
    print(pseudo, "gagne en", score, "→", nouveau)

print()
print(construire_texte_classement(lire_classement(fichier_test)))

Zelda gagne en 6 → [['Zelda', 6]]
Link gagne en 4 → [['Link', 4], ['Zelda', 6]]
Mario gagne en 9 → [['Link', 4], ['Zelda', 6], ['Mario', 9]]
Peach gagne en 5 → [['Link', 4], ['Peach', 5], ['Zelda', 6]]
Toad gagne en 4 → [['Toad', 4], ['Peach', 5], ['Zelda', 6]]

Classement :
1. Toad : 4 essai(s)
2. Peach : 5 essai(s)
3. Zelda : 6 essai(s)


---
## Récapitulatif : notions et leçons

| Notion | Où dans le projet | Leçon / source |
|:---|:---|:---|
| Listes, `append`, `remove`, `index`, tranches `[1:]`, `[-1]` | grille, combinaison, pions mal placés, CSV | Leçon 02B |
| `choice` (module random) | combinaison secrète | Leçon 02B |
| Tableau à 2 dimensions `Grille[l][c]` | identifiants des ronds | leçon sur les tableaux (1ʳᵉ) |
| Copie de liste | `recuperer_proposition`, `calculer_nouveau_classement` | Leçon 02B |
| `try / except FileNotFoundError` | `lire_classement` | Cours du sujet (étape 7) |
| Fichiers `open` / `readlines` / `write` / `split` / `join` | étape 8 | Cours du sujet |
| Tri à bulle | `trier_classement` | Cours du sujet (étape 8) |
| Tkinter : `Canvas`, `Button`, `Entry`, `Label`, `itemconfig`, `config` | interface | Première NSI |

**Conseil pour l'évaluation en classe (sans internet) :** savoir réécrire **seul** `compter_bien_places`, `compter_mal_places` et le tri à bulle. Ce sont les « notions essentielles » les plus probables.